# Chapter 2: Load the transactions and make the first call

This notebook runs the statements in `load-and-call.cypher`, one per cell, so you can see each result as you go. The text above each cell is the comment from the script.

- Run the cells one at a time, from the top.
- Before you start, put `transactions.csv` in the DBMS import folder.

## Prerequisites

Set the following environment variables before launching Jupyter. The URI is the local Neo4j Desktop one:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="neo4j"
```

You also need:

- The plugin JAR in the `plugins` folder of your DBMS, with `dbms.security.procedures.allowlist=jev.*` in `neo4j.conf`.
- For the hosted API only, `server.jvm.additional=-DTYPESAFE_API_KEY=<your-api-key>` in `neo4j.conf`. The notebook never sees the key.

## 1. Install Dependencies

In [1]:
%pip install neo4j==5.28.1 \
             pandas==2.2.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import os
import pandas as pd

from IPython.display import display
from neo4j import GraphDatabase

## 3. Configuration

The connection details come from environment variables set before Jupyter was launched.

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

Credentials set.


## 4. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Helper

`run_cypher` runs one statement. A single map, such as a `jev.decide` result, prints as JSON. Anything else shows as a table.

In [5]:
def run_cypher(query):
    """Run one statement and show the result. A single map prints as JSON, anything
    else as a table. Statements that change the graph also report what changed."""
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query)
        records = [r.data() for r in result]
        summary = result.consume()

    if len(records) == 1 and len(records[0]) == 1 and isinstance(next(iter(records[0].values())), dict):
        print(json.dumps(next(iter(records[0].values())), indent=2, default=str))
    elif records:
        display(pd.DataFrame(records))
    else:
        print("No results.")

    changes = {name: getattr(summary.counters, name, 0) for name in (
        "nodes_created", "nodes_deleted", "relationships_created",
        "relationships_deleted", "properties_set", "constraints_added",
        "constraints_removed")}
    changes = {k: v for k, v in changes.items() if v}
    if changes:
        print("Changes:", ", ".join(f"{k} {v}" for k, v in changes.items()))
    ms = (summary.result_available_after or 0) + (summary.result_consumed_after or 0)
    print(f"Completed after {ms:,} ms")

## 6. The statements

### 1. Is the function installed? Expect one row: `jev.decide`

In [6]:
run_cypher("""
SHOW FUNCTIONS YIELD name WHERE name STARTS WITH 'jev' RETURN name
""")

,name
0,jev.decide


Completed after 119 ms


### 2. Uniqueness constraint on the transaction ID.

In [7]:
run_cypher("""
CREATE CONSTRAINT txn_id IF NOT EXISTS
FOR (t:Transaction) REQUIRE t.txn_id IS UNIQUE
""")

No results.
Completed after 40 ms


### 3. Load the transactions. MERGE means running this twice creates no duplicates.

In [8]:
run_cypher("""
LOAD CSV WITH HEADERS FROM 'file:///transactions.csv' AS row
MERGE (t:Transaction {txn_id: row.txn_id})
SET t.amount                = toFloat(row.amount),
    t.merchant_category     = row.merchant_category,
    t.distance_from_home_km = toFloat(row.distance_from_home_km),
    t.hour_of_day           = toInteger(row.hour_of_day),
    t.account_age_days      = toInteger(row.account_age_days),
    t.prior_flags           = toInteger(row.prior_flags),
    t.txns_last_24h         = toInteger(row.txns_last_24h),
    t.typical_monthly_spend = toInteger(row.typical_monthly_spend),
    t.scenario              = row.scenario,
    t.true_p                = toFloat(row.true_p),
    t.is_fraud              = toInteger(row.is_fraud)
""")

No results.
Changes: properties_set 5500
Completed after 254 ms


### 4. How many did we load? Expect 500 transactions.

In [9]:
run_cypher("""
MATCH (t:Transaction) RETURN count(t) AS transactions
""")

,transactions
0,500


Completed after 23 ms


### 5. The first call. The map projection passes only the eight visible properties, so scenario, true_p and is_fraud never reach Jev. Needs `TYPESAFE_API_KEY` set.

In [10]:
run_cypher("""
MATCH (t:Transaction {txn_id: 'T0001'})
RETURN jev.decide(
  t {.amount, .merchant_category, .distance_from_home_km, .hour_of_day,
     .account_age_days, .prior_flags, .txns_last_24h, .typical_monthly_spend},
  {decision: {
     type: 'choice',
     instructions: 'Based on all available signals, should this transaction be flagged for review or passed through?',
     criteria: {
       Flag: 'Potential fraud. Send for review.',
       Pass: 'Likely legitimate. Process normally.'
     }
  }}
) AS result
""")

{
  "answers": {
    "decision": {
      "type": "choice",
      "choice": "Pass",
      "probabilities": {
        "Pass": 0.92,
        "Flag": 0.08
      },
      "confidence": 0.84
    }
  },
  "error_message": null,
  "metadata": {
    "model": "jev-latest",
    "latency_ms": 408
  }
}
Completed after 522 ms


## 7. Teardown

In [11]:
driver.close()
print("Connection closed.")

Connection closed.
